In [ ]:
import requests
from bs4 import BeautifulSoup
from time import sleep

import pandas as pd
import re

In [ ]:
HEADERS = {"User-Agent": "qangos-scraper/1.0(contact:q@bradtaba.org)"}
API = "https://onepiece.fandom.com/api.php"

character_columns = [
    'Official English Name', 
    'Debut', 
    'Affiliations', 
    'Occupations', 
    'Origin', 
    'Status', 
    'Age', 
    'Birthday', 
    'Height', 
    'Blood Type', 
    'Bounty'
]

In [283]:
def fetch_onepiece(page, sleep_time=2):
    params = {
        "action": "parse",
        "page": page,
        "prop": "text",
        "format": "json"
    }

    try:
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        data = r.json()
        html = data["parse"]["text"]["*"]
        sleep(sleep_time)
    except:
        return False
    else:
        print(f"Page has been successfully fetched! {page}")

    return html


In [ ]:
params = {
        "action": "parse",
        "page": 'Bell-m%C3%A8re',
        "prop": "text",
        "format": "json"
    }


r = requests.get(API, params=params, headers=HEADERS, timeout=30)
data = r.json()
# html = data["parse"]["text"]["*"]
# sleep(1)

In [ ]:
character_page = fetch_onepiece("List_of_Canon_Characters")
soup = BeautifulSoup(character_page, 'html5lib')

char_table = []
for i in soup.tr.find_next_siblings():
    char_data = {
        'name': i.a.text,
        'link': i.a['href'][2:-2].split('/')[-1]
    }
    char_table.append(char_data)
    
char_table = pd.DataFrame(char_table)
char_table.shape

Page has been successfully fetched! List_of_Canon_Characters


In [269]:
status_pattern = r'\(.*?$'
status_regex = re.compile(status_pattern)

name_pattern = r' \(.*?\)'
name_regex = re.compile(name_pattern)

bday_pattern = r'\b[a-z]+\b \d{1,2}'
bday_regex = re.compile(bday_pattern, re.I)

age_pattern = r'\d+'
age_regex = re.compile(age_pattern)

height_pattern = r'\d+ cm'
height_regex = re.compile(height_pattern)

hyperlink_pattern = r'\[\d+\]'
hyperlink_regex = re.compile(hyperlink_pattern)

debut_pattern = r'Chapter (\d+)'
debut_regex = re.compile(debut_pattern, re.I)

In [196]:
armament_page = fetch_onepiece('Haki/Armament_Haki')
armament_soup = BeautifulSoup(armament_page, 'html5lib')

armament_users = []

for i in armament_soup.find_all('small'):
    armament_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Armament_Haki


In [197]:
conq_page = fetch_onepiece('Haki/Supreme_King_Haki')
conq_soup = BeautifulSoup(conq_page, 'html5lib')

conq_users = []

for i in conq_soup.find_all('small'):
    conq_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Supreme_King_Haki


In [198]:
observation_page = fetch_onepiece('Haki/Observation_Haki')
observation_soup = BeautifulSoup(observation_page, 'html5lib')

observation_users = []

for i in observation_soup.find_all('small'):
    observation_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Observation_Haki


In [ ]:
def clean_data(luffy_dict):
    if 'Official English Name' in luffy_dict: 
        luffy_dict['Official English Name'] = name_regex.sub('', luffy_dict['Official English Name'].split(';')[0])

    if 'Affiliations' in luffy_dict:
        luffy_dict['Affiliations'] = luffy_dict['Affiliations'].split(';')[0]

    if 'Occupations' in luffy_dict:
        luffy_dict['Occupations'] = luffy_dict['Occupations'].split(';')[0]

    if 'Bounty' in luffy_dict:
        if isinstance(luffy_dict['Bounty'], int):
            luffy_dict['Bounty'] = luffy_dict['Bounty']
        elif (luffy_dict['Bounty'].lower().startswith('at')) or (luffy_dict['Bounty'].lower()=='unknown'):
            luffy_dict['Bounty'] = 0
        elif luffy_dict['Bounty'].lower().startswith('over'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[1]).split(',')))
        elif luffy_dict['Bounty'].lower().startswith('(') and luffy_dict['Bounty'].lower().endswith(')'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'][1:-1]).split(',')))
        else:
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[0]).split(',')))

    if 'Origin' in luffy_dict:
        luffy_dict['Origin'] = ' '.join(luffy_dict['Origin'].split()[:2])

    if 'Birthday' in luffy_dict:
        luffy_dict['Birthday'] = bday_regex.findall(luffy_dict['Birthday'])[0]

    if 'Age' in luffy_dict:
        ages = age_regex.findall(luffy_dict['Age'].split(';')[-1].strip())
        if ages:
            luffy_dict['Age'] = int(ages[0])
        if luffy_dict['Age'] == 'Multiple Centuries':
            luffy_dict['Age'] = 100

    if 'Height' in luffy_dict:
        luffy_dict['Height'] = int(height_regex.findall(luffy_dict['Height'])[-1][:-3])

    if 'Debut' in luffy_dict:
        luffy_dict['Debut'] = int(debut_regex.findall(luffy_dict['Debut'])[0])

    return luffy_dict

In [229]:
character_info_list = []

In [ ]:
for link in char_table['link'].to_list()[98:]:
    luffy_page = fetch_onepiece(link)
    if not luffy_page:
        print(f'skip: {link}. index {int(char_table[char_table['link']==link].index[0])}')
        print('Next:', int(char_table[char_table['link']==link].index[0])+1)
        continue
    luffy_soup = BeautifulSoup(luffy_page, 'html5lib')

    luffy_dict = {}
    for i in luffy_soup.find_all(class_='pi-data-label pi-secondary-font'):
        if i.text[:-1] in character_columns:
            luffy_dict[i.text[:-1]] = hyperlink_regex.sub(' ', i.find_next_sibling().text).strip()

    fruit_data = luffy_soup.find_all('span', class_='pi-data-label pi-secondary-font')
    if fruit_data:
        luffy_dict['Devil Fruit'] = 1
        luffy_dict['Fruit Type'] = hyperlink_regex.sub(' ', fruit_data[-1].find_next_sibling().text).strip()
    else:
        luffy_dict['Devil Fruit'] = 0

    if 'Official English Name' in luffy_dict:
        luffy_dict['Observation Haki'] = 1 if luffy_dict['Official English Name'] in observation_users else 0
        luffy_dict['Armament Haki'] = 1 if luffy_dict['Official English Name'] in armament_users else 0
        luffy_dict['Conquerers Haki'] = 1 if luffy_dict['Official English Name'] in conq_users else 0
    else:
        luffy_dict['Observation Haki'] = 0
        luffy_dict['Armament Haki'] = 0
        luffy_dict['Conquerers Haki'] = 0

    character_info_list.append(clean_data(luffy_dict))
    print('Next:', int(char_table[char_table['link']==link].index[0])+1)

skip: Ayes%C3%A9_Mar. index 52
Next: 53
Page has been successfully fetched! Azuki
Next: 54
Page has been successfully fetched! Babanuki
Next: 55
Page has been successfully fetched! Babe
Next: 56
Page has been successfully fetched! Baburu
Next: 57
Page has been successfully fetched! Baby_5
Next: 58
Page has been successfully fetched! Bacura
Next: 59
Page has been successfully fetched! Baggaley
Next: 60
Page has been successfully fetched! Bakezo
Next: 61
skip: Ban_Dedessin%C3%A9e. index 61
Next: 62
Page has been successfully fetched! Banchi
Next: 63
Page has been successfully fetched! Banchina
Next: 64
Page has been successfully fetched! Bankuro
Next: 65
Page has been successfully fetched! Banshee
Next: 66
Page has been successfully fetched! Banzaburo
Next: 67
Page has been successfully fetched! Gyoro,_Nin,_and_Bao
Next: 68
Page has been successfully fetched! Bao_Huang
Next: 69
Page has been successfully fetched! Barbel
Next: 70
Page has been successfully fetched! Bariete
Next: 71
Page h

IndexError: list index out of range

In [287]:
luffy_dict

{'Official English Name': 'Biblo',
 'Debut': 'Chapter 1134',
 'Affiliations': 'Owl Library',
 'Occupations': 'Chief Librarian',
 'Status': 'Alive',
 'Age': 'Multiple centuries',
 'Devil Fruit': 1,
 'Fruit Type': 'Paramecia',
 'Observation Haki': 0,
 'Armament Haki': 0,
 'Conquerers Haki': 0}

In [286]:
character_info = pd.DataFrame(character_info_list)
character_info.tail()

,Official English Name,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty
90,Benn Beckman,1.0,Red Hair Pirates,Pirate,Alive,November 9,0,0,0,0,North Blue,50.0,206.0,X,NaN,0.0
91,Bent,1134.0,Walrus School,Student,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
92,Bentham,129.0,Newkama Land,Queen of Newkama Land,Alive,August 15,1,0,0,0,East Blue,30.0,238.0,F,Paramecia,32000000.0
93,Bepo,498.0,Heart Pirates,Pirate,Alive,November 20,0,0,0,0,Grand Line,20.0,240.0,S,NaN,1500.0
94,Bian,717.0,Tontatta Kingdom,Leader of Pink Bee Squad,Alive,NaN,1,0,0,0,NaN,NaN,NaN,NaN,Zoan,NaN
